# 02. Cross sections and event yields
**Lecture 5 · Draft teaching exercise; use instructor-approved samples.**

Use an unweighted MadGraph-only two-lepton sample. A cross section reported by the generator already includes generator cuts. Measure additional analysis acceptance relative to those generated events, not to all physically possible events.

Save your own copy before editing. Run cells from the top. Empty sample selections deliberately do nothing; choose IDs from the available-samples table.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
sample_id = None
luminosities_fb = [0.01, 0.1, 1.0]
pt_min, eta_max = 20., 2.4
if sample_id:
    sample = get_sample(sample_id)
    frame = sample.load()
    if sample.cross_section_pb is None or not sample.generated_events:
        raise ValueError("Choose a sample with a cross section and generated event count.")
    if not np.allclose(weights(frame),1):
        raise ValueError("The binomial uncertainty exercise requires unit-weight events.")
    keep = ((frame.l1_pt > pt_min) & (frame.l2_pt > pt_min)
            & (abs(frame.l1_eta)<eta_max) & (abs(frame.l2_eta)<eta_max))
    n_gen, n_pass = sample.generated_events, int(keep.sum())
    efficiency = n_pass/n_gen
    mc_error = np.sqrt(efficiency*(1-efficiency)/n_gen)
    print("Acceptance:", efficiency, "+/-", mc_error, "(binomial MC uncertainty only)")
    display(pd.DataFrame([{"luminosity_fb": lum, "before_analysis": 1000*lum*sample.cross_section_pb,
                          "after_analysis": 1000*lum*sample.cross_section_pb*efficiency}
                         for lum in luminosities_fb]))

## Questions and submission
1. Explain the factor of 1000 in the yield calculation.
2. Repeat with several cuts; distinguish generator acceptance and analysis acceptance.
3. Compare 1000 and 10000 generated events at fixed luminosity. Which uncertainty should shrink?
4. Explain why the yield prediction is not a random observed count.

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.